In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import matplotlib.dates as mdates
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import pymc as pm
import arviz as az

# Set plotting style
plt.style.use('ggplot')
sns.set_palette("deep")

# Set random seeds for reproducibility
np.random.seed(42)

# Define a function to load data with memory optimization
def load_and_prepare_data(file_path, sample_frac=0.05, max_rows=10000):
    """
    Load and prepare taxi data from a parquet file with memory optimization
    
    Parameters:
    file_path: Path to the parquet file
    sample_frac: Fraction of data to sample (default: 5%)
    max_rows: Maximum number of rows to use (default: 10000)
    
    Returns:
    df: Prepared DataFrame
    """
    try:
        # Read with specific columns only to reduce memory usage
        essential_columns = [
            'tpep_pickup_datetime', 'tpep_dropoff_datetime',
            'trip_distance', 'fare_amount'
        ]
        
        print(f"Loading data from {file_path}...")
        
        try:
            # Try to figure out parquet file schema first
            file_schema = pd.read_parquet(file_path, columns=None, engine='pyarrow', nrows=5)
            print(f"Available columns: {file_schema.columns.tolist()}")
            
            # Filter to only include columns that actually exist in the file
            available_cols = [col for col in essential_columns if col in file_schema.columns]
            if not available_cols:
                print("Warning: None of the essential columns found in file, loading all columns")
                available_cols = None
            
            # Read the parquet file with only available essential columns
            df = pd.read_parquet(file_path, columns=available_cols)
            
        except Exception as e:
            print(f"Error reading with columns specified: {str(e)}")
            # Fallback to reading whole file
            df = pd.read_parquet(file_path)
        
        print(f"Original data shape: {df.shape}")
        
        # Take a random sample to reduce memory footprint
        if sample_frac < 1.0:
            df = df.sample(frac=sample_frac, random_state=42)
        
        # Further limit to max_rows if needed
        if len(df) > max_rows:
            df = df.head(max_rows)
            
        print(f"Sampled data shape: {df.shape}")
        
        # Check if required columns exist
        required_cols = ['tpep_pickup_datetime', 'trip_distance', 'fare_amount']
        missing_cols = [col for col in required_cols if col not in df.columns]
        
        if missing_cols:
            print(f"Warning: Missing columns: {missing_cols}")
            # Create dummy data if needed
            if 'tpep_pickup_datetime' not in df.columns:
                if 'pickup_datetime' in df.columns:
                    df['tpep_pickup_datetime'] = df['pickup_datetime']
                else:
                    # Create a dummy timestamp column
                    base_date = pd.Timestamp('2020-01-01')
                    df['tpep_pickup_datetime'] = [base_date + pd.Timedelta(minutes=i) for i in range(len(df))]
                    
            if 'trip_distance' not in df.columns:
                # Create a random distance column
                df['trip_distance'] = np.random.uniform(1, 10, size=len(df))
                
            if 'fare_amount' not in df.columns:
                # Create a random fare column
                df['fare_amount'] = df.get('trip_distance', np.random.uniform(5, 50, size=len(df))) * np.random.uniform(2, 5)
                
        # Convert datetime columns
        df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
        if 'tpep_dropoff_datetime' in df.columns:
            df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
        
        # Extract datetime components
        df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
        df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
        df['pickup_day'] = df['tpep_pickup_datetime'].dt.day
        df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
        df['pickup_year'] = df['tpep_pickup_datetime'].dt.year
        df['pickup_weekday'] = df['tpep_pickup_datetime'].dt.weekday
        
        # Remove outliers - adjust these filters based on your data
        df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]  # Remove unrealistic distances
        df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]      # Remove unrealistic fares
        
        # Optimize memory usage
        for col in df.select_dtypes(include=['float']).columns:
            df[col] = df[col].astype('float32')
            
        for col in df.select_dtypes(include=['int']).columns:
            df[col] = df[col].astype('int16')
        
        return df
        
    except Exception as e:
        print(f"Error in data loading: {str(e)}")
        # Return a very small dummy dataset if all else fails
        print("Creating a dummy dataset for testing...")
        dates = pd.date_range(start='2020-01-01', periods=100, freq='H')
        dummy_df = pd.DataFrame({
            'tpep_pickup_datetime': dates,
            'trip_distance': np.random.uniform(1, 10, size=100),
            'fare_amount': np.random.uniform(5, 50, size=100)
        })
        dummy_df['pickup_date'] = dummy_df['tpep_pickup_datetime'].dt.date
        dummy_df['pickup_hour'] = dummy_df['tpep_pickup_datetime'].dt.hour
        dummy_df['pickup_day'] = dummy_df['tpep_pickup_datetime'].dt.day
        dummy_df['pickup_month'] = dummy_df['tpep_pickup_datetime'].dt.month
        dummy_df['pickup_year'] = dummy_df['tpep_pickup_datetime'].dt.year
        dummy_df['pickup_weekday'] = dummy_df['tpep_pickup_datetime'].dt.weekday
        return dummy_df

# Function to perform stationarity test
def stationarity_test(time_series, title):
    """
    Perform Augmented Dickey-Fuller test to check for stationarity
    
    Parameters:
    time_series: The time series to test
    title: Title for the test output
    """
    print(f"\nStationarity Test for {title}")
    
    # Ensure we have enough data points
    ts_clean = time_series.dropna()
    if len(ts_clean) < 10:
        print(f"Not enough data points ({len(ts_clean)}) for stationarity test. Skipping.")
        print("-" * 50)
        return
    
    try:
        # Use simpler regression and maxlag settings for short series
        maxlag = min(int(len(ts_clean)/10), 12)  # Adjust maxlag based on series length
        regression = 'c'  # Use constant instead of 'ct' (constant and trend)
        
        result = adfuller(ts_clean, maxlag=maxlag, regression=regression)
        print(f'ADF Statistic: {result[0]:.4f}')
        print(f'p-value: {result[1]:.4f}')
        print('Critical Values:')
        for key, value in result[4].items():
            print(f'\t{key}: {value:.4f}')
        
        if result[1] < 0.05:
            print(f"Result: {title} is stationary (p-value < 0.05)")
        else:
            print(f"Result: {title} is not stationary (p-value >= 0.05)")
    except Exception as e:
        print(f"Error in stationarity test: {str(e)}")
        print(f"Series length: {len(ts_clean)}")
    
    print("-" * 50)

# Function to perform Bayesian structural time series decomposition with memory optimizations
def bsts_decomposition(time_series, title, seasonal_periods=None, n_samples=300, tune=300, 
                       chains=2, cores=1, target_accept=0.9):
    """
    Memory-optimized version that decomposes time series using Bayesian structural time series model with PyMC
    
    Parameters:
    time_series: The time series to decompose
    title: Title for plots
    seasonal_periods: List of seasonal periods to include
    n_samples: Number of posterior samples (reduced)
    tune: Number of tuning samples (reduced)
    chains: Number of MCMC chains (reduced)
    cores: Number of cores (1 to save memory)
    target_accept: Higher value for better stability
    
    Returns:
    components: Dict of component posterior means (no idata to save memory)
    """
    print(f"\nPerforming Bayesian Structural Time Series Decomposition for {title}")
    
    # Convert to numpy array
    y = time_series.values
    n = len(y)
    
    # Standardize data for better sampling
    y_mean = np.mean(y)
    y_std = np.std(y)
    y_standardized = (y - y_mean) / y_std
    
    # Prepare time indices
    t = np.arange(n)
    
    # Set up PyMC model
    with pm.Model() as model:
        # Level (local level) component priors
        sigma_level = pm.HalfNormal("sigma_level", sigma=0.1)
        level_init = pm.Normal("level_init", mu=0, sigma=1)
        
        # Random walk for the level component
        level_innovations = pm.Normal("level_innovations", mu=0, sigma=sigma_level, shape=n-1)
        level = pm.Deterministic("level", pm.math.concatenate([[level_init], level_init + pm.math.cumsum(level_innovations)]))
        
        # Trend (slope) component priors if needed
        if True:  # Set to False to disable trend
            sigma_trend = pm.HalfNormal("sigma_trend", sigma=0.01)
            trend_init = pm.Normal("trend_init", mu=0, sigma=0.1)
            
            # Random walk for trend innovations
            trend_innovations = pm.Normal("trend_innovations", mu=0, sigma=sigma_trend, shape=n-1)
            trend = pm.Deterministic("trend", pm.math.concatenate([[trend_init], trend_init + pm.math.cumsum(trend_innovations)]))
        else:
            trend = pm.Deterministic("trend", pm.math.zeros(n))
        
        # Define seasonal components if needed
        seasonal_components = []
        if seasonal_periods:
            for i, period in enumerate(seasonal_periods):
                # Fourier terms for seasonality (more efficient than dummy variables)
                # Use fewer terms to save memory
                n_fourier_terms = min(period // 4, 4)  # Reduced from 10 to 4
                
                # Fourier series for seasonal components
                fourier_terms = np.zeros((n, 2 * n_fourier_terms))
                for j in range(n_fourier_terms):
                    p = period / (j + 1)
                    fourier_terms[:, 2*j] = np.sin(2 * np.pi * t / p)
                    fourier_terms[:, 2*j+1] = np.cos(2 * np.pi * t / p)
                
                # Prior for seasonal coefficients
                seasonal_coef_name = f"seasonal_coef_{period}"
                seasonal_coef = pm.Normal(seasonal_coef_name, mu=0, sigma=0.1, shape=2*n_fourier_terms)
                
                # Seasonal component
                seasonal_name = f"seasonal_{period}"
                seasonal = pm.Deterministic(seasonal_name, pm.math.dot(fourier_terms, seasonal_coef))
                seasonal_components.append(seasonal)
        
        # Sum up all seasonal components
        if seasonal_components:
            seasonality = pm.Deterministic("seasonality", sum(seasonal_components))
        else:
            seasonality = pm.Deterministic("seasonality", pm.math.zeros(n))
        
        # Observation noise
        sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.1)
        
        # Expected value: trend + level + seasonality
        mu = level + trend + seasonality
        
        # Likelihood
        likelihood = pm.Normal("y", mu=mu, sigma=sigma_obs, observed=y_standardized)
        
        # Sample from the posterior with memory optimizations
        # Use NUTS sampler with memory-efficient options
        idata = pm.sample(
            n_samples, 
            tune=tune, 
            chains=chains,          # Reduced number of chains
            cores=cores,            # Single core to prevent memory issues
            target_accept=target_accept,  # Higher for stability
            return_inferencedata=True
        )
    
    # Calculate posterior means for components
    posterior_means = {
        "level": idata.posterior["level"].mean(dim=["chain", "draw"]).values * y_std + y_mean,
        "trend": idata.posterior["trend"].mean(dim=["chain", "draw"]).values * y_std,
        "seasonality": idata.posterior["seasonality"].mean(dim=["chain", "draw"]).values * y_std,
    }
    
    # Calculate fitted values and residuals
    posterior_means["fitted"] = (posterior_means["level"] + posterior_means["trend"] + 
                                posterior_means["seasonality"])
    posterior_means["residuals"] = y - posterior_means["fitted"]
    
    # Create plots of components - without credible intervals to save memory
    _, axs = plt.subplots(4, 1, figsize=(12, 14))
    
    # Original data with fitted values
    axs[0].plot(time_series.index, y, label="Observed", alpha=0.7)
    axs[0].plot(time_series.index, posterior_means["fitted"], label="Fitted", color="red")
    axs[0].set_title(f"{title} - Observed vs Fitted")
    axs[0].legend()
    axs[0].grid(True)
    
    # Level component
    axs[1].plot(time_series.index, posterior_means["level"], label="Level", color="green")
    axs[1].set_title("Level Component")
    axs[1].grid(True)
    
    # Trend component
    axs[2].plot(time_series.index, posterior_means["trend"], label="Trend", color="blue")
    axs[2].set_title("Trend Component")
    axs[2].grid(True)
    
    # Seasonal component
    axs[3].plot(time_series.index, posterior_means["seasonality"], label="Seasonality", color="purple")
    axs[3].set_title("Seasonal Component")
    axs[3].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Plot residuals
    plt.figure(figsize=(12, 5))
    plt.plot(time_series.index, posterior_means["residuals"], label="Residuals", color="gray")
    plt.title(f"{title} - Residuals")
    plt.grid(True)
    plt.show()
    
    # Calculate and print component statistics
    component_variances = {
        "level": np.var(posterior_means["level"]),
        "trend": np.var(posterior_means["trend"]),
        "seasonality": np.var(posterior_means["seasonality"]),
        "residuals": np.var(posterior_means["residuals"]),
        "total": np.var(y)
    }
    
    # Calculate percentage of variance explained by each component
    variance_explained = {
        k: v / component_variances["total"] * 100 
        for k, v in component_variances.items() if k != "total"
    }
    
    print(f"\nComponent Analysis for {title}:")
    print(f"Total Variance: {component_variances['total']:.4f}")
    for component, explained in variance_explained.items():
        print(f"{component.capitalize()}: {explained:.2f}% of total variance")
    
    # Calculate model fit statistics
    mse = np.mean(posterior_means["residuals"] ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(posterior_means["residuals"]))
    
    print(f"\nModel Fit Statistics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print("-" * 50)
    
    # Clear memory
    del idata
    
    # Check for autocorrelation in residuals - simplified version
    plt.figure(figsize=(12, 5))
    
    # Calculate autocorrelation for multiple lags
    residual_series = pd.Series(posterior_means["residuals"])
    
    # Use fewer lags to save computation
    max_lags = min(30, len(residual_series) // 5)
    lags = range(1, max_lags)
    
    # Calculate autocorrelation for each lag
    autocorr_values = [residual_series.autocorr(lag=lag) for lag in lags]
    
    # Plot autocorrelations
    plt.bar(lags, autocorr_values)
    plt.title(f"{title} - Residual Autocorrelation")
    plt.xlabel("Lag")
    plt.ylabel("Autocorrelation")
    plt.axhline(y=0, color='gray', linestyle='-')
    
    # Add confidence bands at 95% (approximately ±2/√n)
    conf_level = 1.96 / np.sqrt(len(residual_series))
    plt.axhline(y=conf_level, color='r', linestyle='--', alpha=0.7)
    plt.axhline(y=-conf_level, color='r', linestyle='--', alpha=0.7)
    
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    return posterior_means

# Function to analyze seasonal patterns
def analyze_seasonal_patterns(time_series, title, period):
    """
    Analyze seasonal patterns by grouping and averaging data
    
    Parameters:
    time_series: Time series with DatetimeIndex
    title: Title for plots
    period: 'day', 'week', 'month', or 'year'
    """
    print(f"\nAnalyzing {period}ly patterns for {title}")
    
    if period == 'day':
        # Group by hour of day
        grouped = time_series.groupby(time_series.index.hour).mean()
        x_label = 'Hour of Day'
        xticks = np.arange(24)
    elif period == 'week':
        # Group by day of week
        grouped = time_series.groupby(time_series.index.dayofweek).mean()
        x_label = 'Day of Week'
        xticks = np.arange(7)
        plt.xticks(xticks, ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    elif period == 'month':
        # Group by day of month
        grouped = time_series.groupby(time_series.index.day).mean()
        x_label = 'Day of Month'
        xticks = np.arange(1, 32)
    elif period == 'year':
        # Group by month of year
        grouped = time_series.groupby(time_series.index.month).mean()
        x_label = 'Month of Year'
        xticks = np.arange(1, 13)
        plt.xticks(xticks, ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    else:
        raise ValueError("Period must be 'day', 'week', 'month', or 'year'")
    
    # Plot the pattern
    plt.figure(figsize=(12, 5))
    plt.plot(grouped.index, grouped.values, 'o-', linewidth=2, markersize=8)
    plt.title(f"{title} - Average by {x_label}")
    plt.xlabel(x_label)
    plt.ylabel('Average Value')
    plt.grid(True)
    plt.xticks(xticks)
    plt.tight_layout()
    plt.show()
    
    return grouped

# Main execution function with explicit memory management
def main():
    # Load data with sampling to reduce memory usage
    print("Loading and preparing data...")
    df = load_and_prepare_data('/kaggle/input/tsa-taxi-cab/taxi-dataset.parquet', sample_frac=0.05)
    print(f"Loaded data shape: {df.shape}")
    print(f"Data range: {df['tpep_pickup_datetime'].min()} to {df['tpep_pickup_datetime'].max()}")
    
    # Calculate fare rate and clean up outliers
    print("Calculating fare rate...")
    df['fare_rate'] = df['fare_amount'] / df['trip_distance']
    # Remove extreme values
    q1 = df['fare_rate'].quantile(0.01)
    q3 = df['fare_rate'].quantile(0.99)
    df['fare_rate'] = df['fare_rate'].clip(q1, q3)
    
    # Ensure data is sorted by datetime
    df = df.sort_values('tpep_pickup_datetime')
    
    # Set pickup_datetime as index for time series analysis
    df_time = df.set_index('tpep_pickup_datetime').copy()
    
    # Create different aggregations for time series analysis
    print("Creating time series aggregations...")
    
    # Make sure we have enough data before resampling
    if len(df_time) < 100:
        print("Warning: Small dataset detected, reducing aggregation granularity")
    
    # Daily average fare rate - ensure we have enough days for analysis
    daily_fare_rate = df_time.resample('D')['fare_rate'].mean()
    # Remove NaN values instead of interpolating which might create artificial data
    daily_fare_rate = daily_fare_rate.dropna()
    # Keep at most 30 days to manage memory
    daily_fare_rate = daily_fare_rate.tail(30)
    print(f"Daily fare rate data points: {len(daily_fare_rate)}")
    
    # Hourly average fare rate - take just 3 days of hourly data
    hourly_fare_rate = df_time.resample('H')['fare_rate'].mean()
    hourly_fare_rate = hourly_fare_rate.dropna()
    hourly_fare_rate = hourly_fare_rate.tail(24*3)  # Last 3 days
    print(f"Hourly fare rate data points: {len(hourly_fare_rate)}")
    
    # Weekly average fare rate
    weekly_fare_rate = df_time.resample('W')['fare_rate'].mean()
    weekly_fare_rate = weekly_fare_rate.dropna()
    weekly_fare_rate = weekly_fare_rate.tail(10)  # Last 10 weeks
    print(f"Weekly fare rate data points: {len(weekly_fare_rate)}")
    
    # Monthly average fare rate
    monthly_fare_rate = df_time.resample('M')['fare_rate'].mean()
    monthly_fare_rate = monthly_fare_rate.dropna()
    print(f"Monthly fare rate data points: {len(monthly_fare_rate)}")
    
    # Clear the original dataframe to free memory
    del df, df_time
    
    # Perform stationarity tests
    for ts, name in [
        (daily_fare_rate, "Daily Average Fare Rate"),
        (hourly_fare_rate, "Hourly Average Fare Rate"),
        (weekly_fare_rate, "Weekly Average Fare Rate"),
        (monthly_fare_rate, "Monthly Average Fare Rate")
    ]:
        stationarity_test(ts, name)
    
    # Apply Bayesian decomposition to different time scales with reduced samples
    print("\nApplying Bayesian structural time series decomposition...")
    
    # Skip decomposition if not enough data points
    import gc
    
    # Only run decomposition if we have enough data
    if len(daily_fare_rate) >= 14:  # Minimum 2 weeks for meaningful weekly seasonality
        print("Running daily fare rate decomposition...")
        # Daily decomposition with weekly seasonality
        daily_components = bsts_decomposition(
            daily_fare_rate,
            "Daily Average Fare Rate",
            seasonal_periods=[7],  # Weekly seasonality
            n_samples=200,  # Further reduced for memory efficiency
            tune=200,
            chains=2
        )
        # Clear memory
        gc.collect()
    else:
        print("Not enough daily data points for BSTS decomposition.")
    
    if len(hourly_fare_rate) >= 48:  # Minimum 2 days for daily seasonality
        print("Running hourly fare rate decomposition...")
        # Hourly decomposition with daily seasonality
        hourly_components = bsts_decomposition(
            hourly_fare_rate,
            "Hourly Average Fare Rate",
            seasonal_periods=[24],  # Daily seasonality
            n_samples=200,
            tune=200,
            chains=2
        )
        # Clear memory
        gc.collect()
    else:
        print("Not enough hourly data points for BSTS decomposition.")
    
    # Clear memory again
    gc.collect()
    
    # Analyze seasonal patterns only if we have enough data
    print("\nAnalyzing seasonal patterns...")
    
    # Daily patterns for hourly data (if enough hours)
    if len(hourly_fare_rate) >= 24:  # Need at least 24 hours
        daily_pattern = analyze_seasonal_patterns(hourly_fare_rate, "Hourly Fare Rate", "day")
    else:
        print("Not enough hourly data for daily pattern analysis.")
    
    # Weekly patterns for daily data (if enough days)
    if len(daily_fare_rate) >= 7:  # Need at least 7 days
        weekly_pattern = analyze_seasonal_patterns(daily_fare_rate, "Daily Fare Rate", "week")
    else:
        print("Not enough daily data for weekly pattern analysis.")
    
    print("Analysis complete!")

if __name__ == "__main__":
    main()

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 3.3 MB/s eta 0:00:00ta 0:00:01
  Created wheel for cons: filename=cons-0.4.6-py3-none-any.whl size=9100 sha256=b29e51636cb67cdcd2878d731267e2e71574a11a483981126ec266c13e298d3d
  Stored in directory: /Users/tanmay/Library/Caches/pip/wheels/95/8f/45/fe0a5b5e232401da571d514eb545833fbe220993ac8336c94e
  Created wheel for logical-unification: filename=logical_unification-0.4.6-py3-none-any.whl size=13911 sha256=0b369bed0326afbe486e268ad1583b7c07641c6f4b5eafd0224a965223bd497f
  Stored in directory: /Users/tanmay/Library/Caches/pip/wheels/b8/34/a9/c11a21ef1f1b6d2e5ae518dd5d28c0bd2b131c5d6e5d4417c3
  Created wheel for etuples: filename=etuples-0.3.9-py3-none-any.whl size=12618 sha256=9a7aeb3139fd06dfabf1dce5f7bdf6576ba8cda9998978f2dec3246fe796ea94
  Stored in directory: /Users/t

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma_level, level_init, level_innovations, sigma_trend, trend_init, trend_innovations, seasonal_coef_7, sigma_obs]


Output()

Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 29 seconds.
There were 897 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 3 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
